# Predicting Baseball

The game of baseball is a very hard game to predict, as there are numerous factors that go into a game. Factors are almost impossible to model because of how human they are, like the player's mindset going into a game, which can cause players to play better or worse. Additionally, there are factors that add so much complexity to the game that it makes it hard to account for, such as the win blowing at a certain time or the pitches that various pitchers account for. Because of this, it is an interesting challenge to try and attempt to simulate a baseball game by predicting how many runs that are scored by a team. I also wanted some practice incorporating object-oriented programming into modeling and gain further experience with machine learning techniques. The goal of this project is to try and train a model to predict, given a lineup and opposing pitchers, how many runs a team will score. While doing this, I will attempt to limit the number of features to specific stats and do a lineup-based calculation to determine runs.

My general approach to this problem is to first build stat tables for every baseball player and weigh them compared to the league average to account for the vastly different number of plate appearances that exist within rosters. Then, I will create classes for each player to hold their stats, name, player id, and position. I will then create batter and pitcher objects for every player in a lineup. The lineup will be determined from the MLB-StatsAPI, which houses information of who started each game, the pitchers in each game, and the score. I will then get each team's lineup and give that a score using some evaluation of their stats, and do the same for the pitchers and bullpen. Those scores will be what a model takes in and uses to predict the number of runs. Lastly, I will run three different machine learning models on the data to train it and attempt to see what combination of lineup and opposing pitcher scores determines the runs in a game. I will then compare the outcomes of each model. To compare, I will judge them based on the Mean Average Error to see, on average, how far each value is separated.

For the stats I decided to go with, those being strike percent (k_percent), walk percent (bb_percent), wOBA, xwOBA, barrel batted rate (barrel_batted_rate), and hard hit percent (hard_hit_percent), and for the pitchers, whiff percent (whiff_percent). I decided to go with these metrics instead of other descriptive runs due to my approach. By taking stats from 2024 to predict 2025 and 2025 stats to predict 2026 games, this leads to some metrics, like ERA, not being as valuable, as ERA can change wildly between seasons. Take Paul Skenes in 2025 compared to 2026, where in 2025 he had a 1.97 ERA while in 2026 he had a 3.82 ERA. This shows how drastically ERA can change and sometimes a pitcher's value can change. Additionally, batting average was discounted for batters because it does not account for every outcome and the type of hit that happens. By taking hits divided by at bats, that does not account for how good of a hit it was, or if the batter was walked. Because of these factors, I have decided to go with these stats:
- strike percent: Strike percent is a measure of how often a strike happens, so for pitchers, this is a good metric, and for batters, this is a bad metric.
- walk percent: Walk percent is a measure of how often a walk happens. For pitchers, this is a bad outcome, and for batters, this is a good outcome.
- wOBA and xwOBA: These measure how well a batter hits in general, so this is positive for batters and negative for pitchers.
- barrel batted rate and hard hit percent: These are aspects of wOBA that are put in to place emphasis on how well a batter hits
- whiff percent: This is how many of the batters swings result in a miss, so it is a good measure for a pitcher and their placement of pitches.



Some assumptions that are made for this project that are important to describe at this point are 
1) The model will be trained on previous years' stats to predict the current year (i.e., 2024 to predict 2025 and 2025 to predict 2026). This means that 2024 statistics can be used to predict 2025, which means that some values might be overinflated, as one player might have an above-average year, or older players who are out of their prime will still have stats from the year prior. By shrinking towards league average, that attempts to remedy this, but not all of it. This will not fully account for growth or regression between seasons.
2) I will be estimating values for stat weights and trust values; a later model might try to use machine learning to calculate or determine those values. This means the values depend on the weights I choose, which could misrepresent how important stats are.
3) It will use the bullpen of the players that appeared in the game (this means it is trained partly on data that it is trying to predict), so in a real scenario of trying to predict another game, it might be best to have the whole bullpen as the list of available pitchers. This assumption is for simplicity's sake, as in effect, it means that it is using information that is not available at the time of the actual prediction, so this will affect the outcome. This is a convenience assumption to make the model training flow easier. In a practical model, or maybe an extension, it might be best to push past this.
4) Rosters stay the same throughout a game; no pinch hitters are coming in, and every batter plays the full game. This accounts for offensive strength at the start of the game. However, players can be substituted for a variety of reasons, and this model will not account for that. This is also made to simplify the process.
5) Players who do not have statistics from the previous season are given league averages. This means a rookie's skill may not be properly represented. However, this assumption helps make the dataset cleaner, as it gives values to everyone.
6) In generating the dataset, the train_test_split splits everything, 2025 and 2026 combined. This means that MAE may not measure how it performs on an unseen season. Furthermore, the two rows in the train-test split are related, which means the model might test on a team's result and predict that, whilst being trained on a result that is related.

### Imports

First, game data needs to be loaded in and it needs to be added to each player. Using code will help streamline this process, so the MLB-StatsAPI will be used to keep track of the game data. This data will the source of the starters for a team and their batting order. It will also be used to build teams as it has their player ids, which matches how stats are stored. The MLB-StatsAPI will be installed using pip. The cell is raw to stop it from calling after installing it once.



In [148]:
import pandas as pd
import statsapi
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

This project will follow an object oriented approach to designing each team. It will take each player as an object that houses their stats.

### Building the Stats



This is the code for building the stats table for the data. This data comes from Baseball Savant, where stats are listed for most years. To be consistent, stats from 2024 and 2025 will be used to build the players and train the model. csv files have been chosen for this instead of an API so that it is easier to build one table and generate the weighted values at the end. 

In [2]:
batters_2024 = pd.read_csv("2024_batter_stats.csv")
batters_2025 = pd.read_csv("2025_batter_stats.csv")

pitchers_2024 = pd.read_csv("2024_pitcher_stats.csv")
pitchers_2025 = pd.read_csv("2025_pitcher_stats.csv")

fielding_2024 = pd.read_csv("2024_fielding-run-value.csv")
fielding_2025 = pd.read_csv("2025_fielding-run-value.csv")

# removing unnecessariy columns
batters_2024 = batters_2024[["last_name, first_name","player_id","k_percent","bb_percent","woba","xwoba",
                             "barrel_batted_rate","hard_hit_percent","pa"]]
batters_2025 = batters_2025[["last_name, first_name","player_id","k_percent","bb_percent","woba","xwoba",
                             "barrel_batted_rate","hard_hit_percent","pa"]]

# reseting the index to make finding a player easier
batters_2024.index = batters_2024["player_id"]
batters_2024 = batters_2024[["last_name, first_name","k_percent","bb_percent","woba","xwoba",
                             "barrel_batted_rate","hard_hit_percent","pa"]]
batters_2025.index = batters_2025["player_id"]
batters_2025 = batters_2025[["last_name, first_name","k_percent","bb_percent","woba","xwoba",
                             "barrel_batted_rate","hard_hit_percent","pa"]]

# # removing unnecessariy columns
fielding_2024.index = fielding_2024["id"]
fielding_2024 = fielding_2024[["name", "total_runs"]]
fielding_2025.index = fielding_2025["id"]
fielding_2025 = fielding_2025[["name","total_runs"]]



pitchers_2024.index = pitchers_2024["player_id"]
pitchers_2024 = pitchers_2024[["k_percent", "bb_percent", "woba", "xwoba", "barrel_batted_rate", 
                               "hard_hit_percent", "whiff_percent","pa","p_starting_p"]]

pitchers_2025.index = pitchers_2025["player_id"]
pitchers_2025 = pitchers_2025[["k_percent", "bb_percent", "woba", "xwoba", "barrel_batted_rate", 
                               "hard_hit_percent", "whiff_percent","pa","p_starting_p"]]

# Assigning trust values, which is how many plate appearances a batter must have for their stats to count
# Different values are taken for each because there is a difference between the stats, where some stats are more in the capabilities
# of the hitter to hit. These values will be modified to find which ones give a good prediction model
batter_trust = {"k_percent" : 300,
                "bb_percent" : 300,
                "woba" : 300,
                "xwoba" : 300,
                "barrel_batted_rate" : 300,
                "hard_hit_percent" : 300
                }

starting_pitcher_trust = {"k_percent" : 300,
                 "bb_percent" : 300,
                 "woba" : 300,
                 "xwoba" : 300,
                 "barrel_batted_rate" : 300,
                 "hard_hit_percent" : 300,
                 "whiff_percent" : 300
                }

relief_pitcher_trust = {"k_percent" : 300,
                 "bb_percent" : 300,
                 "woba" : 300,
                 "xwoba" : 300,
                 "barrel_batted_rate" : 300,
                 "hard_hit_percent" : 300,
                 "whiff_percent" : 300
                }

This block of code contains the functions that will be used to transform the data tables. The stats need to be weighted to account for some players having a few plate attempts and then with those weights, their stats get scaled. This goes into the third function, which takes the league average and weighs each stat to add to a new table. The product of these three functions is a weighted stats table to account for some players having much more and much less plate attempts.

In [3]:
def weighing_stats(observed, pa, league_average, trust):
    '''
    This function serves to calculate modified values for the batter's stats. This is because some batters have very limited plate appearances,
    so their stats may not reflect how good they actually are. To account for this, averages have been weighted with plate appearances so
    the values fit with the batters actual values. The more at bats they have, the closer their values are to their true stats.

    Parameters:
    observed - this is the batters actual state
    pa - this is how many plate appearances that the batter has
    league_avg - this is the league average for whatever this stat is
    trust - this is how many plate attempts they need for their stat to hold more weight compared to the average. If they are equal, then
    the players stats are as shown

    Returns:
    This then returns a modified value to account for the stat
    '''

    weight = (pa * observed + trust * league_average) / (pa + trust) # this is the formula used to determine weight in the stats, focusing on 
                                                                     # how far they are from the league average in terms of plate appearances
    
    return weight

def calculate_league_average(data, stat):
    '''
    This function serves to calculate the league average for a stat. This function only works for the stats that are in the 
    data table. When calculating the league average, it is weighed by plate attempts to better represent the proportion of 
    players

    Parameters:
    data - This is the dataset the data comes from
    stat - this is the stat that the average is calculated for

    Returns:
    This returns the league average for the stat
    '''
    avg_value = (data[stat] * data["pa"]).sum() / data["pa"].sum() # this is how the weighing formula
    
    return avg_value

def create_weighted_table(data, trust_values):
    '''
    This function then takes the previous functions and makes a weighted table to account for different plate appearances and
    remove some of the chance associated with it

    Parameters:
    data - this is a dataset that will be weighted
    trust_values - these are the values that are needed to trust a player's stats compared to the league average

    Returns:
    A dataset that has modified values to pull towards the league average to account for limited plate attempts
    
    '''
    weighted_table = data.copy() # copy the data so it can be modified without disturbing the original dataframe

    # league averages will be stored to substitute values for players that are not in the system already, such as rookies
    league_avgs = {}

    for stat, trust in trust_values.items(): # then cycle through each stat and weigh their stats
        league_avgs[stat] = calculate_league_average(data, stat)
        weighted_table[stat] = weighing_stats(data[stat], data['pa'], league_avgs[stat], trust_values[stat])

    return weighted_table, league_avgs


### Class Construction

Here is constructing the batter class. Classes have been utilized for this task as they can hold the information such as their position and their name, which will need to be accessed later on in the model.

In [4]:
class Batter:
    '''
    This is a class to hold all of the data for each player to hold their position and their stats
    '''

    def __init__(self, name, position, player_id, offensive_weighted_table, league_averages, defensive_table):
        '''
        This initializes a batter object, this will have the name, the position they are playing, and it will get their stats, both offensively
        and defensively and it will store them in the respective attributes

        Parameters
        name - the name of the player
        position - the position the player plays
        player_id - the MLB id of the player
        offensive_weighted_table - the table from create_weighted_table of their stats
        league_averages - the league average stats for the player
        defenseive_table - the table of defensive stats
        '''

        self.player_id = player_id
        self.name = name
        self.position = position

        # these are stats that will be used
        # One thing to note is that if new stats are added, this dictionary needs to be updated
        self.offensive_stats = {"k_percent" : None,
                "bb_percent" : None,
                "woba" : None,
                "xwoba" : None,
                "barrel_batted_rate" : None,
                "hard_hit_percent" : None
                     }
        
        self.defensive_stats = {"total_runs" : None}
        
        # this builds the offensive stats for the selected stats
        for stat in self.offensive_stats.keys():
            if player_id in offensive_weighted_table.index:
                self.offensive_stats[stat] = offensive_weighted_table[stat].loc[player_id]
            else: # if they are not in the database, they get the league average stats
                self.offensive_stats[stat] = league_averages[stat]

        # this builds the defensive stats for outs above average
        if player_id in defensive_table.index:
            self.defensive_stats["total_runs"] = defensive_table["total_runs"].loc[player_id]
        else: # if they are not in the database, they have 0
            self.defensive_stats["total_runs"] = 0

Here is where the Pitcher class is initialized, which was chosen to be a class for similar reasons as the batter.

In [5]:
class Pitcher:
    def __init__(self, name, player_id, role, pitching_weighted_table, league_averages):
        '''
        This initializes a pitcher object, that has their name, their id, and will generate their stats

        Parameters:
        name - the player's name
        player_id - the MLB id of the player
        pitching_weighted_table - the table from create_weighted_table of their stats
        league_averages - the MLB averages for each stat
        '''
        
        self.name = name
        self.player_id = player_id
        self.role = role

        # these are stats that will be used
        # One thing to note is that if new stats are added, this dictionary needs to be updated
        self.stats = {"k_percent" : None,
                      "bb_percent" : None,
                      "woba" : None,
                      "xwoba" : None,
                      "barrel_batted_rate" : None,
                      "hard_hit_percent" : None,
                      "whiff_percent" : None
                     }

        for stat in self.stats.keys(): # then cycle through each stat and add it to the players
            if player_id in pitching_weighted_table.index: 
                self.stats[stat] = pitching_weighted_table[stat].loc[player_id]
            else:# if they are not in the database, they get the league average
                self.stats[stat] = league_averages[stat]

### Lineup Analysis

The goal of this aspect is to get the starting lineup for a specific game, find the players and the bullpen, and then score them. 

This will involve several functions. First there will be a starting_lineup function which determines the starting lineup for a specific game. There will also need to be functions to get the starting pitcher and the other pitchers. Lastly, there needs to be functions to score the lineup, the starting pitcher, and the bullpen. 

The bullpen and the starting pitchers are separated as the bullpen and starting pitchers tend to play different roles, as the bullpen tends to pitch very few innings, and a starting pitcher pitches for a large chunk of the game.

In [6]:
def starting_lineup(team_box):
    '''
    This takes in a team_box from the statsapi and it generates a lineup from the box score.

    Parameters:
    team_box - the box score for a specific team, so it is the box score of a specific game for the home team

    Returns:
    a list of each starter with their id and their position in their batting order
    '''
    starters = [p for p in team_box["players"].values() #it makes a list of every player in the players category

                # this checks to see if they are pinch hitters or not, as pinch hitters have values that end with 01
                if str(p.get("battingOrder", "")).endswith("00")]
    
    starters.sort(key=lambda p: int(p["battingOrder"]) // 100) # then to get their order, divide by 100 (orders are stored like 100, 200, etc)
    
    return [(p["person"]["id"], p["person"]["fullName"], p["position"]["abbreviation"])
            for p in starters] # this last line returns the starts with their id, full name, and position

def starting_pitcher(team_box):
    '''
    This finds the starting pitcher for a game by taking the first pitcher from the list

    Parameters: 
    team_box is just the box score for a specific team

    Returns
    the starting pitcher with their name and ID
    '''
    pitchers = team_box.get("pitchers", []) 

    # the starting pitcher is the first pitcher in the pitchers list
    pitcher_id = pitchers[0]
    player = team_box["players"][f"ID{pitcher_id}"]

    # then return their name and id
    return {"player_id": pitcher_id,
            "name" : player["person"]["fullName"]
           }

def get_bullpen(team_box):
    '''
    This function gets the relief pitchers that the team uses by getting every pitcher besides the first one

    Inputs:
    team_box - the teams box score

    Output:
    A list of dictionaries containing the relief pitchers ID and name
    '''

    bullpen = []
    
    pitchers = team_box.get("pitchers", [])

    # similar for starting pitcher, every other pitcher besides the first one is put into a list
    for pitcher_id in pitchers[1:]:
        player = team_box["players"][f"ID{pitcher_id}"]
        bullpen.append({"player_id" : pitcher_id,
                        "name" : player["person"]["fullName"]
                       })
    return bullpen

# now adding stats to players
    
def generate_batting_stats(starters, bat_stats, league_average, def_stats):
    '''
    The purpose of this function is to take in the starters and create them as a batter class 
    
    Parameters:
    starters - the list of starters from the starting_lineup
    bat_stats - the batting stats for each player
    league_average - the average for each stat
    def_stats - the defensive stats for each player

    Returns:
    This then will get each stat for each player in the batting order
    '''
    batters = []
    for player_id, name, pos in starters:
        batters.append(Batter(name, pos, player_id, bat_stats, league_average, def_stats))

    return batters


def generate_bullpen_stats(bullpen, pitching_stats, league_average):
    '''
    The purpose of this function is to take the bullpen and add those to the pitchers

    Parameters:
    bullpen - the list of bullpen pitchers
    pitching_stats - the datatable of the pitching stats
    league_average - the league averages for each stat

    Returns:
    This function returns a list of the bullpen that consists of pitcher objects

    '''
    pitchers = []

    for pitcher in bullpen: # cycling through every pitcher
        player_id = pitcher["player_id"]
        name = pitcher["name"]

        pitchers.append(Pitcher(name, player_id, "reliever", pitching_stats, league_average)) # add those pitchers to the list as objects

    return pitchers

order_weights = [2.6, 2.4, 2.2, 2.0, 1.8, 1.6, 1.4, 1.2, 1]

batting_stat_weights = {"k_percent" : -1,
                "bb_percent" : 1.5,
                "woba" : 2.5,
                "xwoba" : 2.0,
                "barrel_batted_rate" : 1.5,
                "hard_hit_percent" : 0.75
                }

# Now I am scoring all of the values to get a numerical value for the teams

def score_lineup(batters, batting_stats, league_average, order_weights, batting_stat_weights):
    '''
    Parameters:
    batters -  the list of batters that in order
    batting_stats - this is the dataset the information comes from
    order_weights - this is the weights for each position in the batting order
    stat_weights - this is how impactful each stat is in deciding score

    Returns:
    A single score value for the batting order
    '''

    score = 0

    for stat, weight in batting_stat_weights.items(): # this cycles through every stat
        # calculating the standard deviation to account for changes from the average
        std = ((((batting_stats[stat] - league_average[stat]) ** 2) * batting_stats["pa"]).sum() / batting_stats["pa"].sum()) ** 0.5 
        for pa, batter in zip(order_weights, batters): # then cycle through every player and add to the score
            score += weight * pa * (batter.offensive_stats[stat] - league_average[stat]) / std

    return score / sum(order_weights) # then take the score for the lineup over the sum of the weights    

pitching_stat_weights = {
    "k_percent": 1.5,
    "bb_percent": -1,
    "woba": -2,
    "xwoba": -2.5,
    "barrel_batted_rate": -1.5,
    "hard_hit_percent": -0.75,
    "whiff_percent": 1
}

def score_starting_pitcher_stats(starter, pitching_stats, league_average, stat_weights):
    '''
    This function scores the starting pitcher for a team, taking in their stats compared to the league average
    
    Parameters:
    the starting pitcher
    pitching_stats - a dataframe with the stats for each pitcher
    league_average - the average for those stats
    stat_weights - how important each stat is

    Returns:
    a score for the starting pitcher
    '''
    score = 0

    for stat, weight in stat_weights.items(): # cycle through every stat and add to the score for the starting pitcher
        std = ((((pitching_stats[stat] - league_average[stat]) ** 2) * pitching_stats["pa"]).sum() / pitching_stats["pa"].sum()) ** 0.5
        score += weight * (starter.stats[stat] - league_average[stat]) / std

    return score


bullpen_stat_weights = {
    "k_percent": 2,
    "bb_percent": -1,
    "woba": -2.5,
    "xwoba": -2,
    "barrel_batted_rate": -1.5,
    "hard_hit_percent": -0.75,
    "whiff_percent": 2
}

def score_bullpen(bullpen, pitching_stats, league_average, bullpen_stat_weights):
    '''
    This function scores the bullpen for a team, taking in their stats compared to the league average
    
    Parameters:
    bullpen - a list containing the bullpen
    pitching_stats - a dataframe with the stats for each pitcher
    league_average - the average for those stats
    bullpen_stat_weights - how important each stat is

    Returns:
    a score for the entire bullpen
    '''
    score = 0

    for pitcher in bullpen: #cycling through every pitcher and using the score_starting_pitcher function
        score += score_starting_pitcher_stats(pitcher, pitching_stats, league_average, bullpen_stat_weights)

    if len(bullpen) != 0:
        return score / len(bullpen)
    else: # if it is an empty bullpen, there was none, so 0
        return 0



The next step is to gather the data. To gather data, the statsapi will be utilized to get the box score for the team, allowing the lineup and the score to be accessed.

### Data Gathering

This process of the model is how the data from the stats api will be turned into features for the model to use. To start off, a game summary is needed to be collected to get the lineups and the boxscore. Then, the features can be built and those functions get used together to establish the dataset that will be used.

In [7]:
def get_game_summary(game_pk):
    '''
    This function generates a game summary for a game, taking in the home team, their line up, and their runs, and also all of those for 
    the other team

    Parameters:
    game_pk - the id number of the game

    Returns:
    a dataframe with the two teams, their lineups, and the score
    '''
    game = statsapi.get("game", {"gamePk": game_pk}) # getting the game

    game_data = game["gameData"]
    live_data = game["liveData"]

    players = live_data["boxscore"]["teams"] # this houses the players for each team

    
    
    teams = game_data["teams"] # getting teams that played
    boxscore = live_data["boxscore"]["teams"] # getting the boxscore for the team 
    linescore = live_data["linescore"]["teams"] # getting the score for the game

    # adding all of that into one dataframe
    return pd.DataFrame({
        "game_pk":[game_pk],
        "home_team": [teams["home"]["name"]],
        "away_team": [teams["away"]["name"]],
        "home_score": [linescore["home"]["runs"]],
        "away_score": [linescore["away"]["runs"]],
        "home_lineup": [starting_lineup(boxscore["home"])],
        "home_pitcher" : [starting_pitcher(boxscore["home"])],
        "home_bullpen" : [get_bullpen(boxscore["home"])],
        "away_lineup": [starting_lineup(boxscore["away"])],
        "away_pitcher" : [starting_pitcher(boxscore["away"])],
        "away_bullpen" : [get_bullpen(boxscore["away"])]
    })



With the code to determine the game summary, it is time to develop the features. The features in the model will be the lineup score, the starting pitcher score, and the bullpen score and those features will attempt to predict the number of runs that will be produced.

In [8]:
def build_features(batters, opposing_pitcher, opposing_bullpen, batting_stats, pitching_stats, batting_league_avg, pitching_league_avg,
                   order_weights, batting_stat_weights, pitching_stat_weights, bullpen_stat_weights):

    '''
    This takes in as inputs basically everything this project has worked towards to generate the features per game, it then
    takes those values and generates the features for the dataframe

    Parameters:
    batters - a list of batters
    opposing_pitcher - the opposing pitcher
    batting_stats - the datatable of batting stats
    pitching_stats - the datatable of pitching stats
    batting_league_avg - the datatable for average stats for batting
    pitching_league_avg - the datatable for average stats for pitching
    order_weights - a list that determines how much each position matters
    batting_stat_weights - a dictionary of how important each stat is for batters
    pitching_stat_weights - a dictionary of how important each stat is for starting pitchers
    bullpen_stat_weights - a dictionary of how important each stat is for the bullpen
    
    Returns:
    It then returns values for all of the features used in the model
    '''
    # scoring the lineup
    lineup_score = score_lineup(
        batters,
        batting_stats,
        batting_league_avg,
        order_weights,
        batting_stat_weights
    )

    # scoring the starting pitcher
    starter_score = score_starting_pitcher_stats(
        opposing_pitcher,
        pitching_stats,
        pitching_league_avg,
        pitching_stat_weights
    )

    # scoring the bullpen
    bullpen_score = score_bullpen(
        opposing_bullpen,
        pitching_stats,
        pitching_league_avg,
        bullpen_stat_weights
    )

    # adding that to a dictionary to return
    features = {
        "lineup_score": lineup_score,
        "opposing_starter_score": starter_score,
        "opposing_bullpen_score": bullpen_score,
    }

    return features

In [9]:
def build_game(game_pk, batting_stats_e, pitching_stats_e, batting_league_avg_e, pitching_league_avg_e,
               fielding_stats_e, batting_stat_weights_e, pitching_stat_weights_e, bullpen_stat_weights_e, order_weights_e):
    '''
    This takes in several inputs and it gets the features for the game that will be played

    Parameters:
    game_pk - the game number
    batting_stats_e - the datatable of batting stats
    pitching_stats_e - the datatable of pitching stats
    batting_league_avg_e - the datatable for average stats for batting
    pitching_league_avg_e - the datatable for average stats for pitching
    order_weights_e - a list that determines how much each position matters
    batting_stat_weights_e - a dictionary of how important each stat is for batters
    pitching_stat_weights_e - a dictionary of how important each stat is for starting pitchers
    bullpen_stat_weights_e - a dictionary of how important each stat is for the bullpen
    
    Returns:
    It then returns values for all of the features used in the model
    '''

    # getting the game summary to add to our build_features function which is called to generate features, this does it across a whole game
    game = get_game_summary(game_pk)
    
    home_lineup_list = game["home_lineup"].iloc[0]
    away_lineup_list = game["away_lineup"].iloc[0]
    
    home_pitcher_data = game["home_pitcher"].iloc[0]
    home_starting_pitcher = Pitcher(home_pitcher_data["name"], home_pitcher_data["player_id"], "starter", pitching_stats_e, pitching_league_avg_e)

    away_pitcher_data = game["away_pitcher"].iloc[0]
    away_starting_pitcher = Pitcher(away_pitcher_data["name"], away_pitcher_data["player_id"], "starter", pitching_stats_e, pitching_league_avg_e)

    home_bullpen = generate_bullpen_stats(game["home_bullpen"].iloc[0],pitching_stats_e,pitching_league_avg_e)

    away_bullpen = generate_bullpen_stats(game["away_bullpen"].iloc[0], pitching_stats_e, pitching_league_avg_e)

    
    
    home_batters = generate_batting_stats(home_lineup_list, batting_stats_e, batting_league_avg_e, fielding_stats_e)
    away_batters = generate_batting_stats(away_lineup_list, batting_stats_e, batting_league_avg_e, fielding_stats_e)

    # calling the build_features function to add it into the game
    home_features = build_features(batters = home_batters, 
                                   opposing_pitcher = away_starting_pitcher,
                                   opposing_bullpen = away_bullpen,
                                   batting_stats = batting_stats_e,
                                   pitching_stats = pitching_stats_e,
                                   batting_league_avg = batting_league_avg_e,
                                   pitching_league_avg = pitching_league_avg_e,
                                   order_weights = order_weights_e,
                                   batting_stat_weights = batting_stat_weights_e,
                                   pitching_stat_weights = pitching_stat_weights_e,
                                   bullpen_stat_weights = bullpen_stat_weights_e)

    away_features = build_features(batters = away_batters, 
                                   opposing_pitcher = home_starting_pitcher,
                                   opposing_bullpen = home_bullpen,
                                   batting_stats = batting_stats_e,
                                   pitching_stats = pitching_stats_e,
                                   batting_league_avg = batting_league_avg_e,
                                   pitching_league_avg = pitching_league_avg_e,
                                   order_weights = order_weights_e,
                                   batting_stat_weights = batting_stat_weights_e,
                                   pitching_stat_weights = pitching_stat_weights_e,
                                   bullpen_stat_weights = bullpen_stat_weights_e)

    return home_features, away_features
    

    

In [10]:
def build_dataset(games_list, batting_stats_g, pitching_stats_g, batting_league_avg_g, pitching_league_avg_g,
                  fielding_stats_g, batting_stat_weights_g, pitching_stat_weights_g, bullpen_stat_weights_g, order_weights_g):
    '''
    Takes in a games_list and calculates the features for each game, for the home and away, and adds them to one training
    dataset that also has the predictor, the number of runs

    Parameters:
    game_list - a list of MLB game ids
    batting_stats_g - the datatable of batting stats
    pitching_stats_g - the datatable of pitching stats
    batting_league_avg_g - the datatable for average stats for batting
    pitching_league_avg_g - the datatable for average stats for pitching
    order_weights_g - a list that determines how much each position matters
    batting_stat_weights_g - a dictionary of how important each stat is for batters
    pitching_stat_weights_g - a dictionary of how important each stat is for starting pitchers
    bullpen_stat_weights_g - a dictionary of how important each stat is for the bullpen
    
    Returns:
    It then returns a dataset of teams, their lineup scores, opposing pitcher score, opposing bullpen score, and the number of runs
    '''

    games_data = []
    skipped_games = []
    # go through each game given and add it in to the games_data with its features
    for game in games_list:
        try:
            home, away = build_game(game, batting_stats_g, pitching_stats_g, batting_league_avg_g, pitching_league_avg_g,
                                fielding_stats_g, batting_stat_weights_g, pitching_stat_weights_g, bullpen_stat_weights_g, order_weights_g)

            game_summary = get_game_summary(game)

            required_columns = [
                "home_team",
                "away_team",
                "home_score",
                "away_score"]

            # the purpose of this is to discount games that are incomplete with their information, allowing the 
            # program to still run despite any unfinished entries
            if game_summary.empty or game_summary[required_columns].isnull().any().any():
                skipped_games.append(game)
                print(f"Skipping game {game}: missing game information")
                continue
        
            games_data.append({"game_pk" : game,
                          "team": game_summary["home_team"].iloc[0],
                          "is_home" : 1,
                          **home,
                          "runs_scored" : game_summary["home_score"].iloc[0]})

            games_data.append({"game_pk" : game,
                      "team": game_summary["away_team"].iloc[0],
                      "is_home" : 0,
                      **away,
                      "runs_scored" : game_summary["away_score"].iloc[0]
            })

        # this chunk of code takes each error and adds it to a list to explain why a game was skipped
        except Exception as e:
            skipped_games.append(game)
            print(f"Skipping game {game}: {e}")
            continue

    # Gives an outlook on building the data
    print(f"Processed {len(games_data) // 2} games.")
    print(f"Skipped {len(skipped_games)} games.")

    return pd.DataFrame(games_data)


        

### Model Construction

Once the features are all collected, a model can then be trained on them. For this project, three different machine learning models have been chosen: Linear regression, random forest regression, and histogram gradient boosting. These three models have been utilized as they each tackle the problem of fitting a line to the data randomly. Linear regression attempts to minimize the error between the expected result and the actual result. Random forest regression combines predictions from several trees with randomized features involved. Histogram gradient boosting builds generations od decision trees where each new generation attempts to reduce error.


Starting out, the games for both 2025 and 2026 need to be gathered. Once again, the stats from 2024 will be used to predict the runs for 2025, and the stats for 2025 will be used to predict the stats for 2026. Because I have the data saved as CSVs, the cell below is the code used to generate the datasets. To save time, the notebook loads the existing datasets instead of regenerating the features. If weights will be updated, these cells below need to be ran again. 

This code contains information that retrieves historical MLB game statistics and builds them into the features dataset using API calls. 



This cell should take around 1923 seconds, depending on the time it takes to call the API.

Because it takes so long, for purposes of reproducibility, the csvs have been converted already, so they just need to be loaded in.

This cell contains the training of the linear regression model. The train-test split includes data from games drawn across those seasons.

In [158]:
training_data_2025 = pd.read_csv("training_data_2025.csv")
training_data_2026 = pd.read_csv("training_data_2026.csv")

features = ["lineup_score", "opposing_starter_score", "opposing_bullpen_score", "is_home"]

data_combined = pd.concat((training_data_2025, training_data_2026))

X = data_combined[features]
y = data_combined["runs_scored"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.1, random_state=549921321)

baseball_predictor_lin = LinearRegression()

baseball_predictor_lin.fit(X_train, y_train)

lin_predictions = baseball_predictor.predict(X_test)

lin_mae = mean_absolute_error(y_test, lin_predictions)

print("Linear Regression MAE:", lin_mae)

Linear Regression MAE: 2.290717057819666


This cell contains the training for the random forest regression model.

In [161]:
baseball_predictor_random_forest = RandomForestRegressor(
    n_estimators=10000,
    min_samples_leaf=10,
    random_state=549921321#,
    #n_jobs=-1
)

baseball_predictor_random_forest.fit(X_train, y_train)

random_forest_predictions = baseball_predictor_random_forest.predict(X_test)

random_forest_mae = mean_absolute_error(y_test, random_forest_predictions)

print("Random Forest MAE:", random_forest_mae)

Random Forest MAE: 2.309526506275192


This cell contains the training for the histogram gradient boosting model.

In [162]:
baseball_predictor_histboost = HistGradientBoostingRegressor(
    max_iter=10000,
    learning_rate=0.01,
    max_leaf_nodes=10,
    min_samples_leaf=10,
    l2_regularization=1,
    random_state=549921321
)

baseball_predictor_histboost.fit(X_train, y_train)

histboost_predictions = baseball_predictor_histboost.predict(X_test)

hist_boost_mae = mean_absolute_error(y_test, histboost_predictions)

print("Gradient Boosting MAE:", boost_mae)

Gradient Boosting MAE: 2.2825915157520997


### Analysis / Interpretation of Results

These results demonstrate that there might be some evidence to suggest that combining player metrics into a team-level feature can play a role in predicting the number of runs that those teams will score. By scoring the lineup, opposing starting pitcher, opposing bullpen, and home vs. away, this model accounts for several possible measures.

After training the models and running it through the games for 2025 and 2026, the difference between the three models is very negligible. Linear regression had a mean average error of 2.29, random forest regression had a mean average error of 2.31, and the gradient boosting regression had a mean average error of 2.28. All three of these are close, and they mean that on average, each model's predictions differ by an average absolute amount of ~2.3 runs. All three of these values are close together and signify that more work needs to be done in determining if one model performed more meaningfully than the others. 

Looking at baseball data, this is very plausible, as there is not really a good way to predict how many runs a team will score, as sometimes a really good pitcher might give up several home runs and that does not get factored into the estimations, as that is very hard to model in. Although historical stats can estimate performance, they alone are unable to predict the runs scored. The goal was to never get an exact model to predict the number of runs, but it was actually to try and make a model to get as close as possible, and for the features used, having a mean average error of around 2.3 is not the best, but with how hard baseball is to predict, I am happy with this. To improve it, I could add more features like slugging percentage or maybe try seeing how ERA and batting average play a role. This project is not finished; I might still go back and modify it, but it is at a finished enough state where it can be shared.

Interpreting these results shows that predicting the number of runs scored is difficult and cannot be broken down into scoring the offense and opposing pitching. There are many other factors, like weather, ballpark, mid-game lineup changes, and if a player is in a slump or not that can affect how well a team will do. All of these factors make baseball so much harder to predict besides the general answer of this team will probably win.

Looking at these results in the context of the assumptions, the statistics used come from the previous season, and they represent player performance in the current season. This does not account for players aging or getting better. Furthermore, the weights I determined may not be rooted in reality, and including only the bullpen pitchers who appeared in the game in the bullpen score limits the accuracy of the model. Despite this, there is still some link between how these stats can be used to predict the number of runs scored.

Overall, this project has shown that historical player statistics could be a starting point to predict runs scored, but further metrics need to be added. Some extensions to this model would be to further add predictors of other stats for each value. Or build off of what I already have and take into account the defense. I have already built the framework for the defense (the batter objects have their defensive stats saved in), so it would need to be added in. Furthermore, addressing the model to have a list of bullpen pitchers and each team's bullpen strength instead of basing it off of the bullpen pitchers that actually pitched at the game. This will help the model better adjust for bullpen strength. Furthermore, the weights that I have estimated could be actually calculated using further machine learning methods. This would help determine which statistics are important when determining the outcome. The main extension to this project would be to actually derive trust and weights for each stat, instead of just guessing. This will most likely improve the accuracy the most.

Works Cited:

Baseballsavant.Com. n.d. “Baseball Savant: Trending MLB Players, Statcast and Visualizations.” Accessed October 22, 2026. https://baseballsavant.mlb.com.

Robb, Todd. 2024. “Home.” GitHub, April 9. https://github.com/toddrob99/MLB-StatsAPI/wiki/Home.